## Google Colab setup

Set `REPO_URL`, `BRANCH`, `RUN_NAME` and `BASE` in the next cell. Open this notebook from the same GitHub branch. The setup mounts Drive, clones code into the runtime and installs dependencies; local execution skips it. Use a fresh runtime after pushing new code. All notebooks in one experiment use the same run name; choose a new name for a fresh experiment. Run preprocessing and feature engineering to completion before model notebooks. The repository README describes the dataset layout and execution order.


In [ ]:
# Configure this cell before running on Google Colab.
import os
import sys
import subprocess
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    REPO_URL = "https://github.com/YOUR_ACCOUNT/YOUR_REPOSITORY.git"
    BRANCH = "YOUR_BRANCH"
    RUN_NAME = "run_01"
    BASE = Path("/content/drive/MyDrive/BT4222")
    CHECKOUT = Path("/content/bt4222_checkout")
    drive.mount("/content/drive")
    if not CHECKOUT.exists():
        subprocess.run([
            "git", "clone", "--branch", BRANCH, "--single-branch",
            REPO_URL, str(CHECKOUT)
        ], check=True)
    active_branch = subprocess.check_output(
        ["git", "-C", str(CHECKOUT), "branch", "--show-current"], text=True
    ).strip()
    if active_branch != BRANCH:
        raise ValueError("Checkout branch differs from BRANCH. Use a fresh Colab runtime.")
    CODE_DIR = (CHECKOUT if (CHECKOUT / "utils/data_utils.py").is_file()
                else CHECKOUT / "bt4222-group12")
    if not (CODE_DIR / "utils/data_utils.py").is_file():
        raise FileNotFoundError("The selected branch does not contain the Group 12 project.")
    os.environ.update({
        "BT4222_BASE": str(BASE),
        "MIND_DATA_DIR": str(BASE / "data"),
        "BT4222_PROCESSED_DIR": str(BASE / "processed"),
        "BT4222_FEATURE_DIR": str(BASE / "features"),
        "BT4222_GLOVE_PATH": str(BASE / "data/glove/glove.6B.300d.txt"),
        "BT4222_OUTPUT_DIR": str(BASE / "runs" / RUN_NAME),
    })
    os.chdir(CODE_DIR)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-r", "requirements.txt"
    ], check=True)
    print("Code:", CODE_DIR)
    print("Features:", os.environ["BT4222_FEATURE_DIR"])
    print("Run outputs:", os.environ["BT4222_OUTPUT_DIR"])


# MLP with Candidate-Conditioned History Attention

Extend the feature-selected MLP with a candidate-dependent representation of the most recent 50 clicked articles. The reference and attention configurations use the same prepared features, scalar normalization, cold-user simulation and validation/test observations.

## Attention mechanism

Encode a candidate and each past click using shared article projections. A learned compatibility network produces $e_{uai}$; mask padded positions and normalize:

$$\alpha_{uai}=\frac{\exp(e_{uai})}{\sum_{j\in H_u}\exp(e_{uaj})},\qquad \mathbf h_{ua}=\sum_{i\in H_u}\alpha_{uai}\mathbf z_i.$$

An empty history has zero attention weights. Project the attended vector as an additional input block and train with the same listwise ranking loss. Candidate-row batching controls memory, not candidate membership. Category/subcategory lookups are 32/32.

## Comparison and fallback

Select the model using validation nDCG@5. Enable a cold-user CTR fallback only if it improves the cold validation cohort. Test evaluates the selected rule; paired bootstrap intervals summarize per-impression differences rather than replacing model selection. Attention weights illustrate learned behavior but are not causal explanations.


In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "utils" / "data_utils.py").is_file()
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from utils.data_utils import project_paths, load_preprocessed, load_table, save_table

if os.environ.get("BT4222_USE_DRIVE") == "1":
    from google.colab import drive

    drive.mount("/content/drive")

PATHS = project_paths(PROJECT_ROOT)
from utils.data_utils import source_fingerprint, validate_features, bind_feature_run

feature_id = validate_features(PATHS)
bind_feature_run(PATHS, feature_id)


In [ ]:
from utils.metrics import ndcg_at_k, history_groups


## 1. Setup
### 0.1 Libraries


In [ ]:
import os, json, time, copy, random
import numpy as np
import pandas as pd
from scipy.stats import rankdata
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as Fnn
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Running on:", device)
FEATURE_DIR = str(PATHS["features"])
OUT_DIR = str(PATHS["outputs"] / "mlp" / "attention")
os.makedirs(OUT_DIR, exist_ok=True)


## 2. Settings
The first block is identical to the prepared input workflow, so the only difference between the two models trained here is the attention block. The second block holds the new settings.


In [ ]:
# Improved MLP configuration
SEED = 42
EMBED_DIM = 64  # every input block is projected to 64 numbers
# Learned category and subcategory vectors use the original 32/32 configuration.
CAT_DIM = 32
SUBCAT_DIM = 32  # Category and subcategory vectors both use 32 dimensions.
HIDDEN = [128, 64, 32]  # layers of the scoring MLP
DROPOUT = 0.2
ID_DROPOUT = 0.1  # randomly hides an article's subcategory during training
LR = 1e-3
WEIGHT_DECAY = 1e-5
BATCH_SIZE = 32  # impressions per training batch (1 click + 4 skipped articles each)
MAX_EPOCHS = 8
PATIENCE = 2  # stop after 2 epochs without a better validation nDCG@5
COLD_SIM_RATE = 0.15  # share of warm users shown as cold users during training
DROP_FEATURES = [
    "diversity_ratio",
    "habit_score",
]  # dropped by 3.3's ablation (config 'no_optional')
NDCG_KS, RECALL_KS = (5, 10), (5, 10)
# Attention configuration
HIST_LEN = 50  # how many of the user's most recent clicks the attention looks at
ATT_HIDDEN = 32  # size of the small network that scores "how related is this click?"
EVAL_ROWS = 8192  # candidates per evaluation batch (keeps memory low on long slates)
SPLITS = ["train", "val", "test"]
pd.set_option("display.float_format", "{:.4f}".format)
pd.set_option("display.width", 200)


## 3. Load the features from the prepared input workflow

| File from feature engineering | What it holds |
|---|---|
| `articles.parquet` | one row per article: id, category, subcategory, has-entity flag |
| `article_content_emb.npy` | MiniLM text vector (384 numbers) per article |
| `kg_entity_emb.npy` | knowledge-graph entity vector (100 numbers) per article |
| `user_profiles.parquet` | one row per distinct click history: length, entropy, ... |
| `user_history_emb.npy` | the user's average clicked-article vector (the improved MLP configuration’s fixed user vector) |
| `user_category_affinity.npy` | share of the user's clicks in each category |
| `{split}/impressions.parquet`, `{split}/groups.parquet`, `{split}/candidates.parquet` | the impressions, and the candidate articles with their features |
| `feature_meta.json` | feature lists, split definition, training means for missing values |


In [ ]:
def load_table(name):
    path = os.path.join(FEATURE_DIR, name)
    return (
        pd.read_parquet(path + ".parquet")
        if os.path.exists(path + ".parquet")
        else pd.read_pickle(path + ".pkl")
    )


with open(os.path.join(FEATURE_DIR, "feature_meta.json")) as f:
    meta = json.load(f)

N_ART, N_CAT, N_SUBCAT = (
    meta["dims"]["n_articles"],
    meta["dims"]["n_cat"],
    meta["dims"]["n_subcat"],
)
SPARSE_MAX = meta["cold_start"][
    "sparse_user_max"
]  # users with 1..SPARSE_MAX-1 clicks are "sparse"
# Articles: lookup tables indexed by news_idx (row 0 = padding)
articles = load_table("articles")
content_emb = np.load(os.path.join(FEATURE_DIR, "article_content_emb.npy")).astype(
    np.float32
)
kg_emb = np.load(os.path.join(FEATURE_DIR, "kg_entity_emb.npy")).astype(np.float32)
has_kg = np.zeros(N_ART, np.float32)
has_kg[articles.news_idx.values] = articles.has_kg_entity.values
art_cat = np.zeros(N_ART, np.int64)
art_cat[articles.news_idx.values] = articles.category_idx.values
art_subcat = np.zeros(N_ART, np.int64)
art_subcat[articles.news_idx.values] = articles.subcategory_idx.values
news_ids = np.empty(N_ART, dtype=object)
news_ids[0] = "<PAD>"
news_ids[articles.news_idx.values] = articles.news_id.values
print("articles             :", len(articles))
print("text vectors         :", content_emb.shape)
print("entity vectors       :", kg_emb.shape)


In [ ]:
# Users: one row per distinct click history (profile_idx 0 = no history)
profiles = load_table("user_profiles").sort_values("profile_idx").reset_index(drop=True)
user_emb = np.load(os.path.join(FEATURE_DIR, "user_history_emb.npy")).astype(np.float32)
user_aff = np.load(os.path.join(FEATURE_DIR, "user_category_affinity.npy")).astype(
    np.float32
)
print("user profiles        :", len(profiles))
print("average user vectors :", user_emb.shape)
print("category affinities  :", user_aff.shape)


In [ ]:
# Impressions and candidates for each split
data = {
    s: {
        "imp": load_table(f"{s}/impressions"),
        "groups": load_table(f"{s}/groups"),
        "cands": load_table(f"{s}/candidates"),
    }
    for s in SPLITS
}

for s in SPLITS:
    days = sorted(int(x) for x in data[s]["imp"].day.unique())
    print(
        f"{s:5s}: {len(data[s]['groups']):>8,} impressions | {len(data[s]['cands']):>10,} candidates | days {days}"
    )

print("Split:", meta["split"]["scheme"])
print("Training negatives:", meta["negative_sampling"]["train"])


## 4. Prepare the inputs
### 2.1 Feature lists
The feature lists come from feature engineering. We use the improved MLP configuration’s selected configuration: every feature group on, `diversity_ratio` and `habit_score` dropped. The assert checks the model contract: none of the excluded features (`slate_length`, `article_recency`, `session_dow`) is used.


In [ ]:
prepared_inputs = pd.read_pickle(Path(FEATURE_DIR) / "mlp_inputs.pkl")
A = prepared_inputs["A"]
cand_mu, cand_sd = prepared_inputs["cand_mu"], prepared_inputs["cand_sd"]
user_mu, user_sd = prepared_inputs["user_mu"], prepared_inputs["user_sd"]
ctx_mu, ctx_sd = prepared_inputs["ctx_mu"], prepared_inputs["ctx_sd"]
user_x_all = prepared_inputs["user_x_all"]


In [ ]:
F = meta["features"]
POP_COLS = F["popularity"]  # article popularity on earlier days
INTER_COLS = F["user_x_article"]  # how well the article matches the user
CAND_COLS = POP_COLS + INTER_COLS
USER_COLS = F["user"]["scalar"] + F["optional"]["user"]
CTX_COLS = F["context"] + F["optional"]["context"]  # time of day
BINARY = {"is_cold_article", "category_match", "has_history"}
used = set(CAND_COLS + USER_COLS + CTX_COLS) | {
    "article_content_emb",
    "kg_entity_emb",
    "category_idx",
    "subcategory_idx",
}
assert not used & set(
    meta["excluded_features"]
), "model contract: an excluded feature is in use"
USE_GROUPS = {
    "article": True,
    "popularity": True,
    "user": True,
    "user_x_article": True,
    "context": True,
}


def make_config(name, groups=None, drop=()):
    return {
        "name": name,
        "groups": dict(groups or USE_GROUPS),
        "drop": sorted(set(drop)),
    }


def active_features(cfg):
    keep = lambda cols: [c for c in cols if c not in cfg["drop"]]
    g = cfg["groups"]
    return {
        "popularity": keep(POP_COLS) if g["popularity"] else [],
        "user_x_article": keep(INTER_COLS) if g["user_x_article"] else [],
        "user": keep(USER_COLS) if g["user"] else [],
        "context": keep(CTX_COLS) if g["context"] else [],
    }


CFG = make_config("no_optional", USE_GROUPS, drop=DROP_FEATURES)

for block, cols in active_features(CFG).items():
    print(f"{block:15s}: {cols}")


## 5. Standardise with training statistics
Continuous features are scaled with the **training** mean and standard deviation only (no information from validation or test). 0/1 flags are left as they are.


In [ ]:
print("standardised user features:", user_x_all.shape)


## 6. Turn each split into flat arrays
Every candidate of every impression goes into one long array; `start[g]` and `n[g]` say where impression `g` begins and how many candidates it has. Training impressions have 5 candidates (1 click + 4 skipped); validation and test keep the full slate.


In [ ]:
assert A["train"]["n"].max() <= 5

for s in SPLITS:
    hl = A[s]["hist_len"]
    print(
        f"{s:5s}: cold users {np.mean(hl == 0):.2%} | sparse users {np.mean((hl > 0) & (hl < SPARSE_MAX)):.2%} | "
        f"candidates that are cold articles {A[s]['is_cold_article'].mean():.2%}"
    )


## 7. Cold-user simulation inputs
As in the improved MLP configuration, 15% of warm users in each training batch are shown to the model as if they had no history. Their inputs are replaced with exactly what a real cold user gets: profile 0 and the training-mean match features. Profile 0 also has an **empty click list** (Section 3), so the attention block learns what to do with cold users too.


In [ ]:
COLD_PROFILE = 0
assert (
    profiles.loc[COLD_PROFILE, "history_length"] == 0
), "profile 0 must be the empty-history profile"
COLD_USER_X = user_x_all[COLD_PROFILE].copy()
COLD_INTER = np.array(
    [(meta["imputation_means"][c] - cand_mu[c]) / cand_sd[c] for c in INTER_COLS],
    np.float32,
)
tr = A["train"]
tr_cold = tr["hist_len"] == 0

if tr_cold.any():
    real_cold_rows = np.repeat(tr_cold, tr["n"])
    assert np.allclose(
        tr["cand_x"][real_cold_rows][:, len(POP_COLS) :], COLD_INTER, atol=1e-4
    )
    assert np.allclose(tr["user_x"][tr_cold], COLD_USER_X, atol=1e-5)

print("Check passed: simulated cold-user inputs = real cold-user inputs")
print(
    f"Real cold users in training: {tr_cold.mean():.2%} -> with simulation about "
    f"{tr_cold.mean() + COLD_SIM_RATE * (~tr_cold).mean():.1%} look cold in each epoch"
)


## 8. Prepared attention histories

Load profile-indexed click histories exported by feature engineering. Rows contain the most recent 50 article indices, with zero padding.


## 9. Match them to feature engineering's impressions


## 10. Build the click-list table `HIST`
One row per profile, 50 columns. Each entry is an article number (`news_idx`), most recent clicks kept, 0 = empty slot. The check at the end confirms that the length of every rebuilt list equals the `history_length` feature engineering computed, so both notebooks agree on what each user clicked.


In [ ]:
HIST = np.load(Path(FEATURE_DIR) / "user_history_indices.npy")
assert HIST.shape == (len(profiles), HIST_LEN)
assert (HIST[COLD_PROFILE] == 0).all()


## 11. Look at one user
A quick look at what the attention block will see: one training user's most recent clicks, with category and title.


In [ ]:
news_raw = load_preprocessed(PATHS)[1].set_index("news_id")


def describe(n):
    """'[category] title' for an article number."""
    row = news_raw.loc[news_ids[n]]
    return f"[{row['category']}] {str(row['title'])[:70]}"


example_prof = A["train"]["prof"][np.argmax(A["train"]["hist_len"] >= 20)]
clicks = HIST[example_prof][HIST[example_prof] > 0]
print(f"Profile {example_prof}: {len(clicks)} clicks in the window. Last 5:")

for n in clicks[-5:]:
    print("  ", describe(n))


## 12. Evaluation metrics
Computed per impression over the full slate, then averaged:
* **MRR**: 1 / rank of the first clicked article
* **nDCG@5, nDCG@10**: how high the clicked articles are in the top 5 / 10 (main metric: nDCG@5, the Spotlight size)
* **Recall@5, @10**: share of clicked articles that make the top 5 / 10
* **AUC**: supplementary, chance a clicked article is scored above a skipped one

Slices: `cold_user` (0 clicks), `sparse_user` (1–4 clicks), `warm_user` (5+), `cold_article` (the clicked article had no clicks on earlier days).


In [ ]:
METRIC_NAMES = (
    ["AUC", "MRR"] + [f"nDCG@{k}" for k in NDCG_KS] + [f"Recall@{k}" for k in RECALL_KS]
)


def impression_metrics(labels, scores):
    order = np.argsort(-scores, kind="stable")
    y = labels[order]
    n_pos = y.sum()
    ranks = np.arange(1, len(y) + 1)
    disc = 1.0 / np.log2(ranks + 1)
    out = {"MRR": float((y / ranks).sum() / n_pos)}
    for k in NDCG_KS:
        ideal = disc[: min(int(n_pos), k)].sum()
        out[f"nDCG@{k}"] = ndcg_at_k(labels, y, k)
    for k in RECALL_KS:
        out[f"Recall@{k}"] = float(y[:k].sum() / n_pos)
    n_neg = len(y) - n_pos
    if n_neg > 0:
        r = rankdata(scores)
        out["AUC"] = float(
            (r[labels == 1].sum() - n_pos * (n_pos + 1) / 2) / (n_pos * n_neg)
        )
    else:
        out["AUC"] = np.nan
    return out


def evaluate(Ad, scores):
    """Returns (table of metrics per slice, metrics per impression)."""
    rows, cold_rows = [], []
    for g in range(len(Ad["n"])):
        s, k = Ad["start"][g], Ad["n"][g]
        lab, sc = Ad["label"][s : s + k], scores[s : s + k]
        rows.append(impression_metrics(lab, sc))
        cold = Ad["is_cold_article"][s : s + k] == 1
        cold_pos = (lab == 1) & cold
        if cold_pos.any():
            keep = ~((lab == 1) & ~cold)
            m = impression_metrics(cold_pos[keep].astype(np.float32), sc[keep])
            m["g"] = g
            cold_rows.append(m)
    per_imp = pd.DataFrame(rows)
    hl = Ad["hist_len"]
    slices = {
        "all": np.ones(len(hl), bool),
        "cold_user": hl == 0,
        "sparse_user": (hl > 0) & (hl < SPARSE_MAX),
        "warm_user": hl >= SPARSE_MAX,
    }
    table = []
    for name, m in slices.items():
        sub = per_imp[m]
        table.append(
            {
                "slice": name,
                "n_impressions": int(m.sum()),
                "n_users": int(pd.unique(Ad["user_id"][m]).size),
                **{k: sub[k].mean() for k in METRIC_NAMES},
            }
        )
    cold_df = pd.DataFrame(cold_rows)
    n_cold_art_users = (
        int(pd.unique(Ad["user_id"][cold_df["g"].values]).size) if len(cold_df) else 0
    )
    table.append(
        {
            "slice": "cold_article",
            "n_impressions": len(cold_df),
            "n_users": n_cold_art_users,
            **{
                k: (cold_df[k].mean() if len(cold_df) else np.nan) for k in METRIC_NAMES
            },
        }
    )
    return pd.DataFrame(table), per_imp


# check against a hand-worked example: clicks at ranks 1 and 3 out of 4
m = impression_metrics(
    np.array([1, 0, 1, 0], np.float32), np.array([0.9, 0.8, 0.7, 0.1])
)
assert (
    abs(m["MRR"] - (1 + 1 / 3) / 2) < 1e-9
    and abs(m["AUC"] - 0.75) < 1e-9
    and m["Recall@5"] == 1.0
)
assert abs(m["nDCG@5"] - (1 + 1 / np.log2(4)) / (1 + 1 / np.log2(3))) < 1e-9
print("Metrics match the hand-worked example")


## 13. Reference MLP architecture

Project feature groups into 64-dimensional blocks and score their concatenation using hidden layers of sizes 128, 64 and 32.


In [ ]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class Proj(nn.Module):
    """Linear layer + LayerNorm: turns one input group into a 64-number block."""

    def __init__(self, d_in, d_out):
        super().__init__()
        self.lin, self.ln = nn.Linear(d_in, d_out), nn.LayerNorm(d_out)

    def forward(self, x):
        return self.ln(self.lin(x))


def make_head(d_in):
    """The scoring MLP: d_in -> 128 -> 64 -> 32 -> 1 score."""
    layers = []
    for h in HIDDEN:
        layers += [nn.Linear(d_in, h), nn.ReLU(), nn.Dropout(DROPOUT)]
        d_in = h
    layers.append(nn.Linear(d_in, 1))
    return nn.Sequential(*layers)


In [ ]:
class MLPRanker(nn.Module):
    """Feature-selected MLP ranker."""

    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        g, act = cfg["groups"], active_features(cfg)
        # lookup tables, stored on the GPU with the model
        for name, arr in {
            "content_emb": content_emb,
            "kg_emb": kg_emb,
            "has_kg": has_kg,
            "art_cat": art_cat,
            "art_subcat": art_subcat,
            "user_emb": user_emb,
            "user_aff": user_aff,
        }.items():
            self.register_buffer(name, torch.as_tensor(arr), persistent=False)
        idx = {
            "pop_idx": [CAND_COLS.index(c) for c in act["popularity"]],
            "inter_idx": [CAND_COLS.index(c) for c in act["user_x_article"]],
            "user_idx": [USER_COLS.index(c) for c in act["user"]],
            "ctx_idx": [CTX_COLS.index(c) for c in act["context"]],
        }
        for name, cols in idx.items():
            self.register_buffer(
                name, torch.as_tensor(cols, dtype=torch.long), persistent=False
            )
        self.on = {
            "article": g["article"],
            "popularity": len(idx["pop_idx"]) > 0,
            "user_x_article": len(idx["inter_idx"]) > 0,
            "user": g["user"],
            "user_scalar": len(idx["user_idx"]) > 0,
            "context": len(idx["ctx_idx"]) > 0,
        }
        d = EMBED_DIM
        self.cat_emb = nn.Embedding(N_CAT, CAT_DIM, padding_idx=0)
        n_blocks = 0
        if self.on["article"]:
            self.subcat_emb = nn.Embedding(N_SUBCAT, SUBCAT_DIM, padding_idx=0)
            self.text_proj = Proj(content_emb.shape[1], d)  # MiniLM text 384 -> 64
            self.kg_proj = Proj(kg_emb.shape[1] + 1, d)  # entities 100 + flag -> 64
            self.cat_proj = Proj(
                CAT_DIM + SUBCAT_DIM, d
            )  # category + subcategory -> 64
            n_blocks += 3
        if self.on["popularity"]:
            self.pop_proj = Proj(len(idx["pop_idx"]), d)
            n_blocks += 1
        if self.on["user"]:
            self.user_text_proj = Proj(
                user_emb.shape[1], d
            )  # average clicked-article vector -> 64
            self.user_cat_proj = Proj(CAT_DIM, d)  # category preferences -> 64
            n_blocks += 2
            if self.on["user_scalar"]:
                self.user_num_proj = Proj(len(idx["user_idx"]), d)
                n_blocks += 1
        if self.on["user_x_article"]:
            self.inter_proj = Proj(len(idx["inter_idx"]), d)
            n_blocks += 1
        if self.on["context"]:
            self.ctx_proj = Proj(len(idx["ctx_idx"]), d)
            n_blocks += 1
        self.n_blocks = n_blocks
        self.mlp = make_head(n_blocks * d)

    def _id_dropout(self, idx):
        if self.training and ID_DROPOUT > 0:
            drop = (torch.rand(idx.shape, device=idx.device) < ID_DROPOUT) & (idx > 1)
            idx = idx.masked_fill(drop, 1)
        return idx

    def blocks(self, news, cand_x, prof, user_x, ctx_x):
        """All of 3.3's 64-number input blocks, by name, in 3.3's order."""
        L = news.shape[1]
        per_user = lambda t: t.unsqueeze(1).expand(
            -1, L, -1
        )  # same user block for every candidate
        b = {}
        if self.on["article"]:
            b["text"] = self.text_proj(self.content_emb[news])
            b["kg"] = self.kg_proj(
                torch.cat([self.kg_emb[news], self.has_kg[news].unsqueeze(-1)], -1)
            )
            cat, sub = self._id_dropout(self.art_cat[news]), self._id_dropout(
                self.art_subcat[news]
            )
            b["cat"] = self.cat_proj(
                torch.cat([self.cat_emb(cat), self.subcat_emb(sub)], -1)
            )
        if self.on["popularity"]:
            b["pop"] = self.pop_proj(cand_x.index_select(-1, self.pop_idx))
        if self.on["user_x_article"]:
            b["inter"] = self.inter_proj(cand_x.index_select(-1, self.inter_idx))
        if self.on["user"]:
            b["user_text"] = per_user(self.user_text_proj(self.user_emb[prof]))
            b["user_cat"] = per_user(
                self.user_cat_proj(self.user_aff[prof] @ self.cat_emb.weight)
            )
            if self.on["user_scalar"]:
                b["user_num"] = per_user(
                    self.user_num_proj(user_x.index_select(-1, self.user_idx))
                )
        if self.on["context"]:
            b["ctx"] = per_user(self.ctx_proj(ctx_x.index_select(-1, self.ctx_idx)))
        return b

    def forward(self, news, mask, cand_x, prof, user_x, ctx_x):
        b = self.blocks(news, cand_x, prof, user_x, ctx_x)
        score = self.mlp(torch.cat(list(b.values()), -1)).squeeze(-1)
        return score.masked_fill(~mask, float("-inf"))


## 14. The attention block (new)
For one candidate article and the user's past clicks, all already turned into 64-number vectors:

1. **Score each click.** A small network looks at the candidate `q` and one click `k` and outputs a relevance score
   `score(q, k) = w · ReLU(W_q q + W_k k + W_p (q × k) + b)`.
   The `q × k` term (element by element) is large when the two articles share features, which is how "same kind of story" gets a high score. This is the scoring unit from DIN (Deep Interest Network, Zhou et al., 2018). It is learned, not hand-set.
2. **Turn scores into weights.** A softmax over the user's clicks gives weights that add up to 1. Empty slots (padding) get weight 0.
3. **Weighted average.** The user vector for this candidate = Σ weight × click vector.

A user with no clicks gets all-zero weights, so this block is the same constant for every candidate and the model ranks them using its other inputs, as the improved MLP configuration does for cold users.


In [ ]:
class ClickAttention(nn.Module):
    """Candidate-aware attention over a user's past clicks (DIN-style scoring unit)."""

    def __init__(self, d, hidden):
        super().__init__()
        self.w_q = nn.Linear(d, hidden)  # looks at the candidate
        self.w_k = nn.Linear(d, hidden, bias=False)  # looks at the past click
        self.w_p = nn.Linear(
            d, hidden, bias=False
        )  # looks at candidate x click (shared features)
        self.out = nn.Linear(hidden, 1)  # -> one relevance score

    def forward(self, q, k, valid):
        """
        q:     candidates   [batch, n_candidates, d]
        k:     past clicks  [batch, n_clicks, d]
        valid: real click?  [batch, n_clicks]  (False = padding)
        returns user vector per candidate [batch, n_candidates, d] and weights [batch, n_candidates, n_clicks]
        """
        q4, k4 = q.unsqueeze(2), k.unsqueeze(1)  # line candidates up against clicks
        hidden = torch.relu(self.w_q(q4) + self.w_k(k4) + self.w_p(q4 * k4))
        scores = self.out(hidden).squeeze(-1)  # [batch, n_candidates, n_clicks]
        valid = valid.unsqueeze(1)
        scores = scores.masked_fill(~valid, -1e4)  # padding -> very low score
        weights = (
            torch.softmax(scores, dim=-1) * valid
        )  # add up to 1; no clicks -> all 0
        user_vec = weights @ k  # weighted average of the clicks
        return user_vec, weights


**Quick check on toy numbers.** One user with 3 real clicks and 2 empty slots, scored against 2 candidates. The block is untrained, so the weights themselves mean nothing yet; this only checks the mechanics: one weight per click, each row adds up to 1, and the empty slots get 0.


In [ ]:
torch.manual_seed(0)
toy = ClickAttention(d=8, hidden=4)
q = torch.randn(1, 2, 8)  # 2 candidates
k = torch.randn(1, 5, 8)  # 5 click slots
valid = torch.tensor([[True, True, True, False, False]])  # last 2 are padding
user_vec, w = toy(q, k, valid)
print("user vector per candidate:", tuple(user_vec.shape))
print("weights (rows = candidates, columns = clicks):")
print(w[0].detach().numpy().round(3))
print("row sums:", w[0].sum(-1).detach().numpy().round(3))


## 15. the improved MLP configuration’s MLP + attention
* Every article (the candidate and each past click) is turned into one 64-number vector the same way, by reusing the improved MLP configuration’s own text and category projections: `article vector = text block + category block`. Using the same encoder for both is what makes a candidate and a past click comparable.
* `ClickAttention` turns the user's last 50 clicks into a user vector for this candidate.
* That vector goes through a `Proj` like every other block and is added as a 10th block, so the scoring MLP sees 640 numbers instead of 576. Everything else, including the improved MLP configuration’s average user vector, stays.


In [ ]:
class MLPWithAttention(MLPRanker):
    def __init__(self, cfg):
        super().__init__(cfg)
        self.register_buffer(
            "hist", torch.as_tensor(HIST), persistent=False
        )  # click list per profile
        self.attention = ClickAttention(EMBED_DIM, ATT_HIDDEN)
        self.user_attn_proj = Proj(EMBED_DIM, EMBED_DIM)
        self.mlp = make_head((self.n_blocks + 1) * EMBED_DIM)  # one more block

    def encode_article(self, news):
        """Article -> 64 numbers, using 3.3's text and category projections."""
        cat = torch.cat(
            [self.cat_emb(self.art_cat[news]), self.subcat_emb(self.art_subcat[news])],
            -1,
        )
        return self.text_proj(self.content_emb[news]) + self.cat_proj(cat)

    def click_weights(self, news, prof):
        """Attention weights for candidates `news` and user profiles `prof` (used in Section 9)."""
        clicks = self.hist[prof]
        _, w = self.attention(
            self.encode_article(news), self.encode_article(clicks), clicks > 0
        )
        return w

    def forward(self, news, mask, cand_x, prof, user_x, ctx_x):
        b = self.blocks(news, cand_x, prof, user_x, ctx_x)
        clicks = self.hist[prof]  # [batch, 50] past clicks
        q = b["text"] + b["cat"]  # the candidates, encoded
        k = self.encode_article(clicks)  # the clicks, encoded the same way
        user_vec, _ = self.attention(q, k, clicks > 0)
        b["user_attn"] = self.user_attn_proj(user_vec)
        score = self.mlp(torch.cat(list(b.values()), -1)).squeeze(-1)
        return score.masked_fill(~mask, float("-inf"))


def build_model(name):
    model = MLPRanker(CFG) if name == "mlp_improved" else MLPWithAttention(CFG)
    return model.to(device)


for name in ["mlp_improved", "mlp_attn"]:
    m = build_model(name)
    print(
        f"{name:9s}: {sum(p.numel() for p in m.parameters()):>8,} parameters | scoring MLP input = {m.mlp[0].in_features} numbers"
    )

del m


## 16. Training setup
### 6.1 Batches
`batch_from` pads impressions in a batch to the same length and, during training, applies the cold-user simulation. Setting a user to profile 0 also empties their click list, because `HIST[0]` is all zeros. Evaluation batches are capped at `EVAL_ROWS` candidates (instead of the improved MLP configuration’s fixed number of impressions) so attention fits in memory on long test slates. This only changes how scores are batched, not the scores.


In [ ]:
def batch_from(Ad, idx, cold_rate=0.0, rng=None):
    idx = np.asarray(idx)
    n, st = Ad["n"][idx], Ad["start"][idx]
    L = int(n.max())
    pos = np.arange(L)
    valid = pos[None, :] < n[:, None]
    rows = np.where(valid, st[:, None] + pos[None, :], 0)
    cand_x = np.where(valid[..., None], Ad["cand_x"][rows], 0.0).astype(np.float32)
    prof, user_x = Ad["prof"][idx].copy(), Ad["user_x"][idx].copy()
    if cold_rate > 0:
        sim = (rng.random(len(idx)) < cold_rate) & (Ad["hist_len"][idx] > 0)
        prof[sim] = COLD_PROFILE
        user_x[sim] = COLD_USER_X
        cand_x[sim, :, len(POP_COLS) :] = COLD_INTER
    to = lambda a, dt=None: torch.as_tensor(a if dt is None else a.astype(dt))
    return {
        "news": to(np.where(valid, Ad["news"][rows], 0)),
        "mask": to(valid),
        "label": to(np.where(valid, Ad["label"][rows], 0.0), np.float32),
        "cand_x": to(cand_x),
        "prof": to(prof),
        "user_x": to(user_x),
        "ctx_x": to(Ad["ctx_x"][idx]),
    }


def train_loader(Ad, seed, cold_rate=COLD_SIM_RATE):
    g = torch.Generator().manual_seed(seed)
    rng = np.random.default_rng(seed + 10_000)
    return DataLoader(
        np.arange(len(Ad["n"])),
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=g,
        collate_fn=lambda idx: batch_from(Ad, idx, cold_rate, rng),
    )


def eval_batches(Ad, max_rows=EVAL_ROWS):
    order = np.argsort(
        Ad["n"], kind="stable"
    )  # similar slate lengths together -> little padding
    batches, cur = [], []
    for g in order:
        if cur and (len(cur) + 1) * Ad["n"][g] > max_rows:
            batches.append(np.array(cur))
            cur = []
        cur.append(g)
    batches.append(np.array(cur))
    return batches


## 17. Loss and prediction
**Listwise softmax loss:** within each impression, a softmax over the candidates' scores, and the loss is −log(probability given to the clicked article). The model is rewarded for putting the click above the other articles in the same impression, which is what ranking needs.


In [ ]:
def forward(model, b):
    b = {k: v.to(device, non_blocking=True) for k, v in b.items()}
    return (
        model(b["news"], b["mask"], b["cand_x"], b["prof"], b["user_x"], b["ctx_x"]),
        b,
    )


def listwise_loss(scores, b):
    logp = Fnn.log_softmax(scores, dim=1).masked_fill(~b["mask"], 0.0)
    return (-(logp * b["label"]).sum(1) / b["label"].sum(1).clamp(min=1)).mean()


@torch.no_grad()
def predict(model, Ad):
    """One score per candidate, in the same order as Ad['news']."""
    model.eval()
    out = np.empty(len(Ad["news"]), np.float32)
    for idx in eval_batches(Ad):
        s, _ = forward(model, batch_from(Ad, idx))
        s = s.float().cpu().numpy()
        for j, g in enumerate(idx):
            st, k = Ad["start"][g], Ad["n"][g]
            out[st : st + k] = s[j, :k]
    return out


## 18. Training loop
Adam, up to 8 epochs. After each epoch the model is scored on the full validation slates; the epoch with the best validation nDCG@5 is kept, and training stops after 2 epochs without improvement. The test set is never used here.


In [ ]:
def train_model(name, seed=SEED):
    set_seed(seed)
    model = build_model(name)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    history, best, best_state, bad = [], -np.inf, None, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        t0 = time.time()
        total, count = 0.0, 0
        for b in train_loader(A["train"], seed + epoch):
            scores, b = forward(model, b)
            loss = listwise_loss(scores, b)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * len(b["prof"])
            count += len(b["prof"])
        val = (
            evaluate(A["val"], predict(model, A["val"]))[0]
            .set_index("slice")
            .loc["all"]
        )
        history.append(
            {
                "epoch": epoch,
                "train_loss": total / count,
                **{f"val_{k}": val[k] for k in METRIC_NAMES},
                "seconds": time.time() - t0,
            }
        )
        print(
            f"  epoch {epoch} | loss {total / count:.4f} | val nDCG@5 {val['nDCG@5']:.4f} | "
            f"val MRR {val['MRR']:.4f} | {time.time() - t0:.0f}s"
        )
        if val["nDCG@5"] > best:
            best, best_state, bad = val["nDCG@5"], copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= PATIENCE:
                print("  early stopping")
                break
    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)


RUN_DIR = os.path.join(OUT_DIR, "runs")
os.makedirs(RUN_DIR, exist_ok=True)


def train_or_load(name):
    """Train the model, or reload it if this exact run was saved before (e.g. after a Colab disconnect)."""
    settings = json.dumps(
        {
            "model": name,
            "feature_id": feature_id,
            "cat_dim": CAT_DIM,
            "subcat_dim": SUBCAT_DIM,
            "seed": SEED,
            "hidden": HIDDEN,
            "lr": LR,
            "epochs": MAX_EPOCHS,
            "cold_sim": COLD_SIM_RATE,
            "hist_len": HIST_LEN,
            "att_hidden": ATT_HIDDEN,
        },
        sort_keys=True,
    )
    weights, hist_csv, info = [
        os.path.join(RUN_DIR, f"{name}.{ext}") for ext in ["pt", "csv", "json"]
    ]
    if (
        all(os.path.exists(p) for p in [weights, hist_csv, info])
        and open(info).read() == settings
    ):
        model = build_model(name)
        model.load_state_dict(torch.load(weights, map_location=device))
        print(f"{name}: reloaded saved run")
        return model, pd.read_csv(hist_csv)
    print(f"Training {name}")
    t0 = time.time()
    model, history = train_model(name)
    torch.save(model.state_dict(), weights)
    history.to_csv(hist_csv, index=False)
    open(info, "w").write(settings)
    print(f"{name}: done in {(time.time() - t0) / 60:.1f} min")
    return model, history


## 19. Train
Both models use the same seed, data, settings and cold-user simulation. The only difference is the attention block.
### 7.1 Baseline: the improved MLP configuration’s model


In [ ]:
baseline, baseline_hist = train_or_load("mlp_improved")


## 20. the improved MLP configuration + attention


In [ ]:
attn_model, attn_hist = train_or_load("mlp_attn")


## 21. Training curves
Left: training loss (should go down). Right: validation nDCG@5 per epoch; the best epoch is the one kept.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))

for name, h in [("mlp_improved", baseline_hist), ("mlp_attn", attn_hist)]:
    ax[0].plot(h.epoch, h.train_loss, marker="o", label=name)
    ax[1].plot(h.epoch, h["val_nDCG@5"], marker="o", label=name)

ax[0].set_title("Training loss")
ax[1].set_title("Validation nDCG@5")

for a in ax:
    a.set_xlabel("epoch")
    a.legend()

plt.tight_layout()
plt.show()


## 22. Results
### 8.1 Score validation and test
Each model scores every candidate in the full validation and test slates. `popularity_ctr` (rank by click rate on earlier days, no personalisation) is the reference point.


In [ ]:
MODELS = {"mlp_improved": baseline, "mlp_attn": attn_model}
CTR_COL = CAND_COLS.index("hist_ctr")
scores, per_imp, tables = {}, {}, []

for split in ["val", "test"]:
    for name, model in MODELS.items():
        scores[(name, split)] = predict(model, A[split])
    scores[("popularity_ctr", split)] = A[split]["cand_x"][:, CTR_COL]
    for name in list(MODELS) + ["popularity_ctr"]:
        table, per_imp[(name, split)] = evaluate(A[split], scores[(name, split)])
        table.insert(0, "split", split)
        table.insert(0, "model", name)
        tables.append(table)

results = pd.concat(tables, ignore_index=True)
print("Scored:", sorted({k[0] for k in scores}))


## 23. Overall test results
**Main comparison:** `mlp_attn` against `mlp_improved` on nDCG@5. Both rows use the same test impressions.


In [ ]:
cols = ["mlp_improved", "mlp_attn", "popularity_ctr"]
overall = (
    results[(results.split == "test") & (results.slice == "all")]
    .set_index("model")
    .loc[cols, METRIC_NAMES]
)
overall


## 24. Results by user type (nDCG@5)
Attention needs past clicks to work with, so the gain should show up mostly for `warm_user`. With 0 clicks the block is constant, so `cold_user` should change little.


In [ ]:
order = ["all", "cold_user", "sparse_user", "warm_user", "cold_article"]

for split in ["val", "test"]:
    print(f"\n{split} nDCG@5")
    print(
        results[results.split == split]
        .pivot(index="slice", columns="model", values="nDCG@5")
        .loc[order, cols]
        .to_string()
    )

sizes = results[(results.split == "test") & (results.model == "mlp_improved")][
    ["slice", "n_impressions", "n_users"]
]
print("\nTest impressions / users per slice")
print(sizes.to_string(index=False))


## 25. Pick the final model (on validation only)
1. The model with the higher **validation** nDCG@5 is kept.
2. For users with **no history**, we check on validation whether plain popularity ranks better than the model. If it does, those users are ranked by popularity (`+ fallback`). the improved MLP configuration reported the same fallback as a reference.

Both choices use validation only; the test numbers are just reported afterwards.


In [ ]:
val_all = results[(results.split == "val") & (results.slice == "all")].set_index(
    "model"
)["nDCG@5"]
BEST = val_all[list(MODELS)].idxmax()
val_cold = results[(results.split == "val") & (results.slice == "cold_user")].set_index(
    "model"
)["nDCG@5"]
USE_FALLBACK = bool(val_cold["popularity_ctr"] > val_cold[BEST])
print(f"Best on validation: {BEST}")
print(
    f"Validation, cold users: {BEST} {val_cold[BEST]:.4f} vs popularity {val_cold['popularity_ctr']:.4f} "
    f"-> fallback {'ON' if USE_FALLBACK else 'OFF'}"
)
FINAL = BEST + (" + fallback" if USE_FALLBACK else "")

if USE_FALLBACK:
    for split in ["val", "test"]:
        cold_rows = np.repeat(A[split]["hist_len"] == 0, A[split]["n"])
        scores[(FINAL, split)] = np.where(
            cold_rows, scores[("popularity_ctr", split)], scores[(BEST, split)]
        )
        table, per_imp[(FINAL, split)] = evaluate(A[split], scores[(FINAL, split)])
        table.insert(0, "split", split)
        table.insert(0, "model", FINAL)
        results = pd.concat([results, table], ignore_index=True)

final_test = (
    results[(results.split == "test") & (results.model == FINAL)]
    .set_index("slice")
    .loc[order, METRIC_NAMES]
)
print(f"\nFinal model: {FINAL} (test)")
final_test


## 26. Is the difference real or luck?
A difference of 0.002–0.003 in nDCG@5 can come from which impressions happen to be in the test day. A **paired bootstrap** checks this: resample the test impressions 1,000 times (same impressions for both models) and look at the middle 95% of the difference. If that interval does not include 0, the gain is unlikely to be luck in the test sample.

Limitation: this does not cover training randomness (a different seed gives a slightly different model). Checking that would mean retraining with several seeds.


In [ ]:
def paired_bootstrap(a, b, n_boot=1000, seed=SEED):
    rng = np.random.default_rng(seed)
    diff = (a - b).astype(np.float64)
    boots = np.concatenate(
        [
            diff[rng.integers(0, len(diff), (100, len(diff)))].mean(1)
            for _ in range(n_boot // 100)
        ]
    )
    return diff.mean(), np.percentile(boots, 2.5), np.percentile(boots, 97.5)


hl = A["test"]["hist_len"]
groups = {
    "all": np.ones(len(hl), bool),
    "cold_user": hl == 0,
    "sparse_user": (hl > 0) & (hl < SPARSE_MAX),
    "warm_user": hl >= SPARSE_MAX,
}
rows = []

for new, old in [("mlp_attn", "mlp_improved"), (BEST, "popularity_ctr")]:
    for g, mask in groups.items():
        a = per_imp[(new, "test")]["nDCG@5"].values[mask]
        b = per_imp[(old, "test")]["nDCG@5"].values[mask]
        mean, lo, hi = paired_bootstrap(a, b)
        rows.append(
            {
                "comparison": f"{new} vs {old}",
                "users": g,
                "nDCG@5 difference": mean,
                "95% low": lo,
                "95% high": hi,
                "real (excludes 0)": not (lo <= 0 <= hi),
            }
        )

bootstrap = pd.DataFrame(rows)
bootstrap


## 27. Where does attention help?
### 9.1 By history length
nDCG@5 of `mlp_attn` minus `mlp_improved` on test, per number of past clicks. Above 0 = attention helped.


In [ ]:
gain = (
    per_imp[("mlp_attn", "test")]["nDCG@5"].values
    - per_imp[("mlp_improved", "test")]["nDCG@5"].values
)
buckets = pd.cut(
    A["test"]["hist_len"],
    [-1, 0, 4, 9, 19, 49, np.inf],
    labels=["0", "1-4", "5-9", "10-19", "20-49", "50+"],
)
by_length = (
    pd.DataFrame({"clicks in history": buckets, "gain": gain})
    .groupby("clicks in history", observed=False)["gain"]
    .agg(nDCG5_gain="mean", impressions="size")
)
print(by_length.to_string())
plt.figure(figsize=(6, 3.2))
plt.bar(by_length.index.astype(str), by_length["nDCG5_gain"], color="#2a6fb0")
plt.axhline(0, color="black", lw=0.8)
plt.xlabel("clicks in history")
plt.ylabel("nDCG@5: attention − 3.3")
plt.title("Gain from attention by history length (test)")
plt.tight_layout()
plt.show()


## 28. By how varied a user's reading is
The reason for attention is users with mixed interests: an average blurs them, attention can pick the relevant part. Warm test users are split into three equal groups by `reading_entropy` (low = mostly one category, high = many categories). If the idea holds, the gain should be larger for the high-diversity group.


In [ ]:
At = A["test"]
warm = At["hist_len"] >= SPARSE_MAX
entropy = profiles["reading_entropy"].values[At["prof"]][warm]
third = pd.qcut(
    pd.Series(entropy).rank(method="first"),
    3,
    labels=["low diversity", "medium", "high diversity"],
)
by_diversity = (
    pd.DataFrame(
        {"reading diversity": third.values, "entropy": entropy, "gain": gain[warm]}
    )
    .groupby("reading diversity", observed=False)
    .agg(
        mean_entropy=("entropy", "mean"),
        nDCG5_gain=("gain", "mean"),
        impressions=("gain", "size"),
    )
)
by_diversity


## 29. One user, two candidates
A test user who reads at least 3 categories, and two candidates from different categories in the same impression. For each candidate, the three past clicks with the highest attention weight. If attention works, the top clicks change with the candidate (for example, sports clicks for a sports article). A single average vector cannot do this.


In [ ]:
rng = np.random.default_rng(SEED)
example = None

for g in rng.permutation(len(At["n"])):
    clicks = HIST[At["prof"][g]]
    clicks = clicks[clicks > 0]
    if len(clicks) < 10 or len(np.unique(art_cat[clicks])) < 3:
        continue
    s, k = At["start"][g], At["n"][g]
    pick = []
    for c in At["news"][s : s + k]:
        if art_cat[c] in set(art_cat[clicks]) and art_cat[c] not in [
            art_cat[x] for x in pick
        ]:
            pick.append(c)
        if len(pick) == 2:
            example = (g, pick, clicks)
            break
    if example:
        break

if example is None:
    print("No suitable example found")
else:
    g, pick, clicks = example
    attn_model.eval()
    with torch.no_grad():
        w = (
            attn_model.click_weights(
                torch.as_tensor([pick], device=device),
                torch.as_tensor([At["prof"][g]], device=device),
            )[0]
            .cpu()
            .numpy()[:, : len(clicks)]
        )
    mix = (
        pd.Series([news_raw.loc[news_ids[n], "category"] for n in clicks])
        .value_counts()
        .to_dict()
    )
    print(f"User {At['user_id'][g]}: {len(clicks)} clicks, categories {mix}\n")
    for j, c in enumerate(pick):
        print("Candidate:", describe(c))
        for i in np.argsort(-w[j])[:3]:
            print(f"   weight {w[j, i]:.3f}  {describe(clicks[i])}")
        print(f"   (equal weighting would give {1 / len(clicks):.3f} each)\n")


## 30. Save
* `attention_results.csv`: every model × split × user group, all metrics
* `bootstrap_test.csv`, `gain_by_history.csv`, `gain_by_diversity.csv`
* `predictions_test`, `prediction_test.txt`, `topk_test.csv`: the final model's test rankings, same format as the improved MLP configuration
* `run_config.json`: settings used


In [ ]:
results.to_csv(os.path.join(OUT_DIR, "attention_results.csv"), index=False)
bootstrap.to_csv(os.path.join(OUT_DIR, "bootstrap_test.csv"), index=False)
by_length.to_csv(os.path.join(OUT_DIR, "gain_by_history.csv"))
by_diversity.to_csv(os.path.join(OUT_DIR, "gain_by_diversity.csv"))
# test rankings of the final model (same files as 3.3)
At, sc = A["test"], scores[(FINAL, "test")]
gid = np.repeat(np.arange(len(At["n"])), At["n"])
pred = pd.DataFrame(
    {
        "impression_id": At["impression_id"][gid],
        "orig_impression_id": At["orig_impression_id"][gid],
        "source": At["source"][gid],
        "user_id": At["user_id"][gid],
        "news_id": news_ids[At["news"]],
        "slot": At["slot"],
        "label": At["label"].astype(np.int8),
        "score": sc.astype(np.float32),
        "history_length": At["hist_len"][gid],
        "is_cold_article": At["is_cold_article"],
    }
)
pred["rank"] = (
    pred.groupby("impression_id")["score"]
    .rank(ascending=False, method="first")
    .astype(np.int32)
)

try:
    pred.to_parquet(os.path.join(OUT_DIR, "predictions_test.parquet"), index=False)
except ImportError:
    pred.to_pickle(os.path.join(OUT_DIR, "predictions_test.pkl"))

pred = pred.sort_values(["impression_id", "slot"])

with open(os.path.join(OUT_DIR, "prediction_test.txt"), "w") as f:
    for imp_id, r in pred.groupby("impression_id", sort=True)["rank"]:
        f.write(f"{imp_id} [{','.join(map(str, r.values))}]\n")

top10 = (
    pred[pred["rank"] <= 10]
    .sort_values(["impression_id", "rank"])
    .groupby(["impression_id", "user_id"])["news_id"]
    .agg(" ".join)
    .reset_index()
    .rename(columns={"news_id": "top10_news_ids"})
)
top10.to_csv(os.path.join(OUT_DIR, "topk_test.csv"), index=False)
run_config = {
    "notebook": "6_2_mlp_attention",
    "builds_on": ["3_feature_engineering", "6_1_mlp"],
    "final_model": FINAL,
    "selected_on_validation": BEST,
    "cold_user_fallback": USE_FALLBACK,
    "best_epoch": {
        "mlp_improved": int(
            baseline_hist.loc[baseline_hist["val_nDCG@5"].idxmax(), "epoch"]
        ),
        "mlp_attn": int(attn_hist.loc[attn_hist["val_nDCG@5"].idxmax(), "epoch"]),
    },
    "settings": {
        "cat_dim": CAT_DIM,
        "subcat_dim": SUBCAT_DIM,
        "seed": SEED,
        "hidden": HIDDEN,
        "embed_dim": EMBED_DIM,
        "lr": LR,
        "batch_size": BATCH_SIZE,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "cold_sim_rate": COLD_SIM_RATE,
        "hist_len": HIST_LEN,
        "att_hidden": ATT_HIDDEN,
        "dropped_features": DROP_FEATURES,
    },
    "features": active_features(CFG),
    "contract": {
        k: meta[k]
        for k in [
            "split",
            "negative_sampling",
            "cold_start",
            "leakage",
            "excluded_features",
        ]
    },
}

with open(os.path.join(OUT_DIR, "run_config.json"), "w") as f:
    json.dump(run_config, f, indent=2, default=str)

print("Saved to", OUT_DIR)
print(sorted(os.listdir(OUT_DIR)))


## 31. Summary and limitations
**Result (seed 42, test day, full slates).** Adding attention over the last 50 clicks to the prepared input workflow's MLP raises nDCG@5 from 0.3506 to 0.3575 (+0.0069), MRR from 0.3194 to 0.3246 and Recall@5 from 0.4894 to 0.4989. The paired bootstrap puts the nDCG@5 gain between +0.0055 and +0.0082.

**Where it helps.** Warm users (+0.0073) and sparse users (+0.0060). The gain appears from the first few clicks and is similar for narrow and broad readers.

**Where it does not.** Cold users (no clicks to attend to; popularity ranks them best on test) and cold articles (slightly worse than the prepared input workflow's model).

**Limitations.**
* One training seed. the prepared input workflow's model scored 0.3600 in the prepared input workflow (CPU) and 0.3506 when re-trained here (GPU), a gap larger than the attention gain; the bootstrap covers test-sample luck but not training randomness.
* Attention sees only the last 50 clicks; 75% of training impressions have their whole history within 50. Older clicks reach the model only through the prepared input workflow's average vector.
* Attention compares article content (text and category), not which other users clicked what.
* One test day (15 Nov 2019) of MIND-small, a one-week sample of MSN News.
